In [1]:

from pathlib import Path
import sys

# Project root directory
PROJECT_ROOT = Path(r"D:\Drone-research")

# Define important project locations
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Select yesterday's flight log
SELECTED_LOG = RAW_DIR / "2026-07-02 18-37-42.bin"

# Verify our environment
print("Python version:", sys.version.split()[0])
print("Project exists:", PROJECT_ROOT.exists())
print("Raw data directory exists:", RAW_DIR.exists())
print("Selected flight log exists:", SELECTED_LOG.exists())

if SELECTED_LOG.exists():
    print(
        "Selected log size:",
        round(SELECTED_LOG.stat().st_size / (1024 ** 2), 2),
        "MiB"
    )


Python version: 3.11.17
Project exists: True
Raw data directory exists: True
Selected flight log exists: True
Selected log size: 15.14 MiB


In [2]:

import pandas as pd
from pymavlink import mavutil

def extract_telemetry(log_path, message_type, instance=None):
    """
    Extract telemetry from an ArduPilot BIN log.
    """

    log = mavutil.mavlink_connection(str(log_path))
    records = []

    while True:
        # Read every message sequentially
        msg = log.recv_match()

        if msg is None:
            break

        # Filter message type manually
        if msg.get_type() != message_type:
            continue

        data = msg.to_dict()

        # Optional sensor instance filtering
        if instance is not None:
            sensor_id = data.get("I", data.get("IMU"))

            if sensor_id != instance:
                continue

        records.append(data)

    df = pd.DataFrame(records)

    if not df.empty and "TimeUS" in df.columns:
        df = df.sort_values(
            "TimeUS"
        ).reset_index(drop=True)

    return df


In [3]:

att_df = extract_telemetry(
    SELECTED_LOG,
    "ATT"
)

print("Total attitude records:", len(att_df))
print("Available columns:", att_df.columns.tolist())

display(att_df.head())


Total attitude records: 11437
Available columns: ['mavpackettype', 'TimeUS', 'DesRoll', 'Roll', 'DesPitch', 'Pitch', 'DesYaw', 'Yaw', 'ErrRP', 'ErrYaw', 'AEKF']


,mavpackettype,TimeUS,DesRoll,Roll,DesPitch,Pitch,DesYaw,Yaw,ErrRP,ErrYaw,AEKF
0,ATT,23845225,0.0,1.24,0.0,-0.47,0.03,0.03,0.0,1.0,3
1,ATT,23945401,0.0,1.24,0.0,-0.47,0.02,0.02,0.0,1.0,3
2,ATT,24047397,0.0,1.24,0.0,-0.47,0.03,0.02,0.0,1.0,3
3,ATT,24147365,0.0,1.24,0.0,-0.47,0.02,0.02,0.0,1.0,3
4,ATT,24247300,0.0,1.25,0.0,-0.47,0.02,0.02,0.0,1.0,3


In [4]:

# Extract telemetry using our reusable function

gps_df = extract_telemetry(SELECTED_LOG, "GPS", instance=0)
imu_df = extract_telemetry(SELECTED_LOG, "IMU", instance=0)
baro_df = extract_telemetry(SELECTED_LOG, "BARO", instance=0)
arm_df = extract_telemetry(SELECTED_LOG, "ARM")

# Display dataset information
datasets = {
    "ATT": att_df,
    "GPS": gps_df,
    "IMU": imu_df,
    "BARO": baro_df,
    "ARM": arm_df
}

for name, df in datasets.items():
    print(
        f"{name}: {len(df)} records, "
        f"{len(df.columns)} columns"
    )


ATT: 11437 records, 11 columns
GPS: 5718 records, 16 columns
IMU: 28591 records, 16 columns
BARO: 11437 records, 11 columns
ARM: 18 records, 6 columns


In [5]:

# Check which sensor instances are present

for name in ["GPS", "IMU", "BARO"]:
    df = datasets[name]

    print(f"\n{name} sensor instances:")

    if "I" in df.columns:
        print(df["I"].value_counts().sort_index())
    else:
        print("No instance column found")



GPS sensor instances:
I
0    5718
Name: count, dtype: int64

IMU sensor instances:
I
0    28591
Name: count, dtype: int64

BARO sensor instances:
I
0    11437
Name: count, dtype: int64


In [6]:

import pandas as pd

quality_reports = []

for name, df in datasets.items():

    # Count missing values
    missing_values = df.isnull().sum().sum()

    # Count duplicate timestamps
    duplicate_times = (
        df["TimeUS"].duplicated().sum()
        if "TimeUS" in df.columns else 0
    )

    # Count invalid timestamps
    invalid_times = (
        (df["TimeUS"] <= 0).sum()
        if "TimeUS" in df.columns else 0
    )

    quality_reports.append({
        "Dataset": name,
        "Records": len(df),
        "Missing Values": missing_values,
        "Duplicate Timestamps": duplicate_times,
        "Invalid Timestamps": invalid_times
    })

quality_df = pd.DataFrame(quality_reports)

display(quality_df)


,Dataset,Records,Missing Values,Duplicate Timestamps,Invalid Timestamps
0,ATT,11437,0,0,0
1,GPS,5718,0,0,0
2,IMU,28591,0,0,0
3,BARO,11437,0,0,0
4,ARM,18,0,0,0


In [7]:

from pathlib import Path

# Create the processed data directory
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Use a unique name for our selected flight log
log_id = SELECTED_LOG.stem.replace(" ", "_")

# Export each telemetry dataset
for name, df in datasets.items():

    output_path = (
        PROCESSED_DIR /
        f"{log_id}_{name.lower()}.csv"
    )

    df.to_csv(output_path, index=False)

    print(
        f"Saved: {output_path.name} "
        f"({len(df)} records)"
    )

print("\nAll telemetry datasets exported successfully!")


Saved: 2026-07-02_18-37-42_att.csv (11437 records)
Saved: 2026-07-02_18-37-42_gps.csv (5718 records)
Saved: 2026-07-02_18-37-42_imu.csv (28591 records)
Saved: 2026-07-02_18-37-42_baro.csv (11437 records)
Saved: 2026-07-02_18-37-42_arm.csv (18 records)

All telemetry datasets exported successfully!


In [8]:

import sys
from pathlib import Path

# Allow the notebook to access the src folder
sys.path.insert(0, str(PROJECT_ROOT))

# Import our new parser
from src.log_parser import parse_flight_log

# Extract multiple telemetry types in one pass
selected_types = ["ATT", "GPS", "IMU", "BARO", "ARM"]

parsed_data = parse_flight_log(
    SELECTED_LOG,
    message_types=selected_types
)

# Display results
print("Single-pass parser results:\n")

for name in selected_types:
    df = parsed_data.get(name)

    if df is not None:
        print(f"{name}: {len(df)} records")
    else:
        print(f"{name}: Not found")


Single-pass parser results:

ATT: 11437 records
GPS: 5718 records
IMU: 28591 records
BARO: 11437 records
ARM: 18 records


In [9]:

from pathlib import Path
import pandas as pd
from src.log_parser import parse_flight_log

RAW_DIR = PROJECT_ROOT / "data" / "raw"

log_files = sorted(
    path for path in RAW_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() == ".bin"
)

summary_records = []
failed_logs = []

for index, log_path in enumerate(log_files, start=1):
    try:
        # Parse only the messages needed for inventory
        data = parse_flight_log(
            log_path,
            message_types=["ATT", "GPS", "ARM"]
        )

        att = data.get("ATT", pd.DataFrame())
        gps = data.get("GPS", pd.DataFrame())
        arm = data.get("ARM", pd.DataFrame())

        all_times = []

        for df in [att, gps, arm]:
            if not df.empty and "TimeUS" in df.columns:
                all_times.extend([
                    df["TimeUS"].min(),
                    df["TimeUS"].max()
                ])

        duration = (
            (max(all_times) - min(all_times)) / 1_000_000
            if all_times else None
        )

        summary_records.append({
            "Filename": log_path.name,
            "Size_MiB": round(
                log_path.stat().st_size / (1024 ** 2), 2
            ),
            "ATT_Count": len(att),
            "GPS_Count": len(gps),
            "ARM_Count": len(arm),
            "Observed_Duration_s": duration
        })

        print(f"[{index}/{len(log_files)}] OK: {log_path.name}")

    except Exception as exc:
        failed_logs.append({
            "Filename": log_path.name,
            "Error": str(exc)
        })
        print(f"[{index}/{len(log_files)}] FAILED: {log_path.name}")

summary_df = pd.DataFrame(summary_records)

print("\nSuccessfully processed:", len(summary_df))
print("Failed logs:", len(failed_logs))

display(summary_df.head())


[1/102] OK: 1 01-01-1980 05-30-00 AM.bin
[2/102] OK: 10 01-01-1980 05-30-00 AM.bin
[3/102] OK: 100 01-01-1980 05-30-00 AM.bin
[4/102] OK: 101 01-01-1980 05-30-00 AM.bin
[5/102] OK: 102 01-01-1980 05-30-00 AM.bin
[6/102] OK: 11 01-01-1980 05-30-00 AM.bin
[7/102] OK: 12 01-01-1980 05-30-00 AM.bin
[8/102] OK: 13 01-01-1980 05-30-00 AM.bin
[9/102] OK: 14 01-01-1980 05-30-00 AM.bin
[10/102] OK: 1980-01-09 18-01-22.bin
[11/102] OK: 1980-01-10 18-20-39.bin
[12/102] OK: 1980-01-10 19-03-03.bin
[13/102] OK: 1980-01-11 11-17-24.bin
[14/102] OK: 1980-01-11 18-45-27.bin
[15/102] OK: 1980-01-11 18-54-46.bin
[16/102] OK: 2 01-01-1980 05-30-00 AM.bin
[17/102] OK: 20 01-01-1980 05-30-00 AM.bin
[18/102] OK: 2026-05-14 18-47-26.bin
[19/102] OK: 2026-05-15 18-42-30.bin
[20/102] OK: 2026-05-19 11-10-00.bin
[21/102] OK: 2026-05-19 11-44-09.bin
[22/102] OK: 2026-05-22 11-00-24.bin
[23/102] OK: 2026-05-22 11-01-15.bin
[24/102] OK: 2026-05-22 11-15-18.bin
[25/102] OK: 2026-05-22 11-28-43.bin
[26/102] OK: 2026

,Filename,Size_MiB,ATT_Count,GPS_Count,ARM_Count,Observed_Duration_s
0,1 01-01-1980 05-30-00 AM.bin,0.50,270,135,0,26.901668
1,10 01-01-1980 05-30-00 AM.bin,0.75,414,207,0,41.302572
2,100 01-01-1980 05-30-00 AM.bin,0.28,140,0,0,13.901679
3,101 01-01-1980 05-30-00 AM.bin,0.29,144,0,0,14.302099
4,102 01-01-1980 05-30-00 AM.bin,0.29,142,0,0,14.140713


In [10]:

from pathlib import Path

# Create output directory
reports_dir = PROJECT_ROOT / "outputs" / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# Save our batch inventory
inventory_path = reports_dir / "flight_log_inventory.csv"

summary_df.to_csv(
    inventory_path,
    index=False
)

print("Inventory saved successfully!")
print("Location:", inventory_path)
print("Total logs recorded:", len(summary_df))


Inventory saved successfully!
Location: D:\Drone-research\outputs\reports\flight_log_inventory.csv
Total logs recorded: 102


In [11]:

from src.log_parser import parse_flight_log

# Extract all available message types
all_data = parse_flight_log(
    SELECTED_LOG,
    message_types=None
)

print("Total message types extracted:", len(all_data))

print("\nMessage types and record counts:\n")

for name, df in sorted(all_data.items()):
    print(f"{name}: {len(df)} records")


Total message types extracted: 55

Message types and record counts:

AHR2: 11437 records
ARM: 18 records
ATT: 11437 records
BARO: 11437 records
CMD: 1 records
CTRL: 11436 records
CTUN: 11437 records
D32: 9 records
DSF: 1141 records
DU32: 1144 records
ERR: 7 records
EV: 70 records
FILE: 10397 records
FMT: 171 records
FMTU: 171 records
GPA: 5718 records
GPS: 5718 records
IMU: 28591 records
MAG: 22866 records
MODE: 16 records
MOTB: 11437 records
MSG: 28 records
MULT: 15 records
ORGN: 20 records
PARM: 1224 records
PM: 114 records
POS: 11437 records
POWR: 11433 records
PSCD: 11437 records
PSCE: 11150 records
PSCN: 11150 records
RATE: 11437 records
RCIN: 11437 records
RCO2: 11437 records
RCOU: 11437 records
SRTL: 136 records
STAK: 11344 records
TERR: 1144 records
UBX2: 229 records
UNIT: 34 records
VER: 1 records
VIBE: 11437 records
XKF1: 11437 records
XKF2: 11437 records
XKF3: 11437 records
XKF4: 11437 records
XKF5: 11437 records
XKFM: 1011 records
XKFS: 11437 records
XKQ: 11437 records
XKT:

In [14]:

import pandas as pd
from pathlib import Path

# Create a folder for Excel reports
excel_dir = PROJECT_ROOT / "outputs" / "excel_reports"
excel_dir.mkdir(parents=True, exist_ok=True)

# Define output file
excel_path = (
    excel_dir / "2026-07-02_18-37-42_full_telemetry.xlsx"
)

# Export all extracted message types
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    for msg_type, df in sorted(all_data.items()):

        # Excel supports a maximum of 1,048,576 rows
        if len(df) > 1_048_575:
            print(f"Skipped {msg_type}: exceeds Excel row limit")
            continue

        # Convert nested or binary values to readable strings
        export_df = df.copy()

        for col in export_df.columns:
            export_df[col] = export_df[col].map(
                lambda value: repr(value)
                if isinstance(value, (bytes, list, dict, tuple))
                else value
            )

        export_df.to_excel(
            writer,
            sheet_name=msg_type[:31],
            index=False
        )

        print(f"Exported {msg_type}: {len(df)} rows")

print("\nExcel workbook created successfully!")
print("Location:", excel_path)


Exported AHR2: 11437 rows
Exported ARM: 18 rows
Exported ATT: 11437 rows
Exported BARO: 11437 rows
Exported CMD: 1 rows
Exported CTRL: 11436 rows
Exported CTUN: 11437 rows
Exported D32: 9 rows
Exported DSF: 1141 rows
Exported DU32: 1144 rows
Exported ERR: 7 rows
Exported EV: 70 rows
Exported FILE: 10397 rows
Exported FMT: 171 rows
Exported FMTU: 171 rows
Exported GPA: 5718 rows
Exported GPS: 5718 rows
Exported IMU: 28591 rows
Exported MAG: 22866 rows
Exported MODE: 16 rows
Exported MOTB: 11437 rows
Exported MSG: 28 rows
Exported MULT: 15 rows
Exported ORGN: 20 rows
Exported PARM: 1224 rows
Exported PM: 114 rows
Exported POS: 11437 rows
Exported POWR: 11433 rows
Exported PSCD: 11437 rows
Exported PSCE: 11150 rows
Exported PSCN: 11150 rows
Exported RATE: 11437 rows
Exported RCIN: 11437 rows
Exported RCO2: 11437 rows
Exported RCOU: 11437 rows
Exported SRTL: 136 rows
Exported STAK: 11344 rows
Exported TERR: 1144 rows
Exported UBX2: 229 rows
Exported UNIT: 34 rows
Exported VER: 1 rows
Expor